# 04 — UIT Inference (LHP-guided Feature Selection + ITM Rerank)

**Faithful to paper Algorithm 1 (Feature Selection):**

1. Encode gallery → **token-level** image_embeds (cache fp16 on GPU)
2. Encode queries → token-level text_embeds
3. ITC similarity (cosine of pooled feats) = baseline for candidates outside top-K
4. **Load `scores_lhp.pt`** from 03 → top-K candidates per query (paper Algorithm 1)
5. **Cross-Encoder + ITM Head** on (query, candidate_image) pairs for each top-K → P(match)
6. Replace top-K positions in the final score matrix with ITM probability

**Why this matters:** the paper trains UIT with 4 losses (ITC + **ITM** + MLM + 0.1356·MIM) — ignoring ITM at inference means discarding 1/4 of the training signal. LHP acting as the "guide" to select candidates for UIT is the core point of the paper's iterative ensemble logic.

**Inputs:**
- `checkpoints/uit/checkpoint_best.pth` (from 02)
- `features/lhp_peg14/scores_lhp.pt` (from 03 — **required**)
- `manifests/{gallery,query,val_split,val_zeroshot_scene}_manifest.parquet`

**Outputs:**
- `features/uit/{gallery,queries,val,val_zs}.npz` (ITC-pooled embeddings)
- `features/uit/gallery_tokens.pt` (gallery image_embeds token-level cache, optional persist)
- `features/uit/scores_uit.pt` (final = ITC outside top-K + ITM inside top-K)
- `features/uit/scores_uit_itc_only.pt` (ITC baseline for ablation)
- `features/uit/scores_{val,val_zs}.pt`

**Hyperparams:**
- `ITM_TOPK = 256` (paper Algorithm 1 uses small top-K; K=256 balances accuracy/cost)
- `ITM_FUSION_ALPHA = 0.4` (weight ITC vs ITM in top-K: final = α·ITC + (1-α)·ITM)
- Gallery token cache fp16 ≈ 36773 × ~50 tokens × 1024 dim ≈ 3.7GB VRAM (fits 80GB)

**ETA:** ~45-60 min (gallery encode ~15min, ITC ~3min, ITM rerank ~30min).

In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

NOTEBOOK_DIR = Path('.').resolve()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from aic_colab_utils import (
    setup_aic2026_environment, select_a100_device,
    sync_chunk_to_drive, wait_for_pending_syncs, maybe_clear_cuda,
    save_npz_atomic,
)

PATHS = setup_aic2026_environment()
INPUT_ROOT = PATHS['input_root']; MANIFEST_DIR = PATHS['manifest_dir']
LOCAL_ROOT = PATHS['local_root']; OUTPUT_ROOT = PATHS['output_root']
DRIVE_OUTPUT_ROOT = PATHS['drive_output_root']

UIT_CKPT_DIR = OUTPUT_ROOT / 'checkpoints' / 'uit'
UIT_FEAT_DIR = OUTPUT_ROOT / 'features' / 'uit'
LHP_FEAT_DIR = OUTPUT_ROOT / 'features' / 'lhp_peg14'
UIT_FEAT_DIR.mkdir(parents=True, exist_ok=True)

UIT_REPO = LOCAL_ROOT / 'aio_repo'
UIT_CODE = UIT_REPO / 'uit' / 'cmp'
device = select_a100_device()

# Locate best checkpoint
ckpt_best = UIT_CKPT_DIR / 'checkpoint_best.pth'
if not ckpt_best.exists():
    eps = sorted(UIT_CKPT_DIR.glob('checkpoint_*.pth'), key=lambda p: int(p.stem.split('_')[1]))
    assert eps, 'No UIT checkpoint found. Run 02_uit_train.ipynb first.'
    ckpt_best = eps[-1]
print('UIT checkpoint:', ckpt_best)

# Sanity: LHP scores must exist for Algorithm 1
LHP_SCORES = LHP_FEAT_DIR / 'scores_lhp.pt'
assert LHP_SCORES.exists(), f'{LHP_SCORES} missing. Run 03_lhp_peg14_train.ipynb first.'
print('LHP guidance:', LHP_SCORES)

In [ ]:
def pip_install(*p):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *p])
try:
    import timm, torchscale, transformers, yacs  # noqa
except Exception:
    pip_install('timm==0.6.13', 'torchscale==0.3.0', 'transformers==4.47.1', 'yacs', 'ruamel.yaml')

import torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm.auto import tqdm

gallery_df = pd.read_parquet(MANIFEST_DIR / 'gallery_manifest.parquet')
query_df = pd.read_parquet(MANIFEST_DIR / 'query_manifest.parquet')
val_df = pd.read_parquet(MANIFEST_DIR / 'val_split.parquet')
val_zs_df = pd.read_parquet(MANIFEST_DIR / 'val_zeroshot_scene.parquet')

def remap_path(p):
    if not isinstance(p, str) or not p: return p
    if p.startswith(str(INPUT_ROOT)): return p
    for anchor in ('annotation/', 'train/imgs_', 'name-masked_test-set/', 'gallery/'):
        i = p.find(anchor)
        if i >= 0: return str(INPUT_ROOT / p[i:])
    return p

for df_ in (gallery_df, val_df, val_zs_df):
    if 'image_path' in df_.columns:
        df_['image_path'] = df_['image_path'].map(remap_path)

print('gallery:', len(gallery_df), 'query:', len(query_df), 'val:', len(val_df), 'val_zs:', len(val_zs_df))

In [ ]:
# === Load UIT model from checkpoint ===
if str(UIT_CODE) not in sys.path:
    sys.path.insert(0, str(UIT_CODE))

import yaml
from torchvision import transforms
from torchvision.transforms import InterpolationMode
from transformers import BertTokenizer

cfg_path = UIT_CODE / 'configs' / 'cmp_a100_80gb.yaml'
if not cfg_path.exists():
    cfg_path = UIT_CODE / 'configs' / 'cmp.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

os.chdir(UIT_CODE)
from models.cmp import CMP

model = CMP(config=cfg)
state = torch.load(ckpt_best, map_location='cpu')
state_dict = state['model'] if isinstance(state, dict) and 'model' in state else state
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print(f'load_state_dict: missing={len(missing)} unexpected={len(unexpected)}')
model = model.to(device).eval()

bert_path = cfg.get('text_encoder', 'bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained(bert_path if Path(bert_path).exists() else 'bert-base-uncased')
MAX_TOKENS = cfg.get('max_tokens', 56)

uit_norm = transforms.Normalize((0.38901278, 0.3651612, 0.34836376),
                                 (0.24344306, 0.23738699, 0.23368555))
uit_transform = transforms.Compose([
    transforms.Resize((cfg['h'], cfg['w']), interpolation=InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    uit_norm,
])
print('Model loaded. h×w:', cfg['h'], cfg['w'], '| max_tokens:', MAX_TOKENS)

In [ ]:
# === Encode gallery: store BOTH pooled ITC feat AND token-level image_embeds ===
#
# Token cache required for cross-encoder ITM. For gallery 36773 × ~50 tokens × 1024 dim fp16
# ≈ 3.7GB VRAM — still fits on A100 80GB.
IMG_BATCH = 128
amp_dtype = torch.bfloat16

class _ImageDS(Dataset):
    def __init__(self, df, id_col):
        self.paths = df['image_path'].astype(str).tolist()
        self.ids = df[id_col].astype(str).tolist()
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            pil = Image.open(self.paths[i]).convert('RGB'); ok = True
        except Exception:
            pil = Image.new('RGB', (cfg['h'], cfg['w'])); ok = False
        return uit_transform(pil), self.ids[i], self.paths[i], ok

@torch.inference_mode()
def encode_images_full(df, id_col, keep_tokens=True):
    """Returns:
        pooled_emb (N, D) fp16 L2-normed     — for ITC
        tokens_list (list of (N_tokens, D_v) tensors fp16) — for ITM cross-encoder
        atts_list (matching attention masks)
        ids, paths, ok
    """
    loader = DataLoader(_ImageDS(df, id_col), batch_size=IMG_BATCH,
                        num_workers=12, pin_memory=True,
                        persistent_workers=True, prefetch_factor=4)
    pooled, tokens, atts, ids, paths, ok_arr = [], [], [], [], [], []
    for x, b_ids, b_paths, b_ok in tqdm(loader, desc=f'uit-img-encode {id_col}'):
        x = x.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            image_embeds, image_atts = model.get_vision_embeds(x)
            image_feat = model.get_image_feat(image_embeds)
            image_feat = F.normalize(image_feat.float(), dim=-1)
        pooled.append(image_feat.cpu().numpy().astype('float16'))
        if keep_tokens:
            tokens.append(image_embeds.half().cpu())  # offload to CPU; will re-upload per query batch
            atts.append(image_atts.cpu())
        ids.extend(b_ids); paths.extend(b_paths); ok_arr.extend([bool(v) for v in b_ok])
    pooled = np.concatenate(pooled, 0)
    if keep_tokens:
        tokens = torch.cat(tokens, dim=0)   # (N, T_v, D_v) fp16 on CPU
        atts = torch.cat(atts, dim=0)       # (N, T_v) long on CPU
    return pooled, tokens, atts, np.array(ids), np.array(paths), np.array(ok_arr)

print('Encoding gallery (pooled + token-level)…')
g_pooled, g_tokens, g_atts, g_ids, g_paths, g_ok = encode_images_full(gallery_df, 'gallery_id', keep_tokens=True)
print('gallery: pooled', g_pooled.shape, '| tokens', tuple(g_tokens.shape), '| atts', tuple(g_atts.shape))
save_npz_atomic(UIT_FEAT_DIR / 'gallery.npz', ids=g_ids, paths=g_paths, embeddings=g_pooled, ok=g_ok)

In [ ]:
# === Encode queries: pooled ITC feat + token-level text_embeds (cached for ITM) ===
TXT_BATCH = 128

@torch.inference_mode()
def encode_texts_full(ids, texts, keep_tokens=True):
    pooled, tok_embeds, tok_atts, tok_ids_buf = [], [], [], []
    for s in tqdm(range(0, len(texts), TXT_BATCH), desc='uit-txt'):
        batch = list(texts[s:s + TXT_BATCH])
        tok = tokenizer(batch, padding='max_length', truncation=True,
                        max_length=MAX_TOKENS, return_tensors='pt').to(device)
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            text_embeds = model.get_text_embeds(tok.input_ids, tok.attention_mask)
            text_feat = model.get_text_feat(text_embeds)
            text_feat = F.normalize(text_feat.float(), dim=-1)
        pooled.append(text_feat.cpu().numpy().astype('float16'))
        if keep_tokens:
            tok_embeds.append(text_embeds.half().cpu())
            tok_atts.append(tok.attention_mask.cpu())
    pooled = np.concatenate(pooled, 0)
    if keep_tokens:
        tok_embeds = torch.cat(tok_embeds, dim=0)  # (Nq, T_t, D_t)
        tok_atts = torch.cat(tok_atts, dim=0)
    return pooled, tok_embeds, tok_atts, np.array([str(x) for x in ids])

print('Encoding queries (pooled + token-level)…')
q_pooled, q_tokens, q_atts, q_ids = encode_texts_full(
    query_df['query_index'].astype(str).tolist(),
    query_df['caption'].astype(str).tolist(),
    keep_tokens=True,
)
print('queries: pooled', q_pooled.shape, '| tokens', tuple(q_tokens.shape))
save_npz_atomic(UIT_FEAT_DIR / 'queries.npz', ids=q_ids, embeddings=q_pooled)

In [ ]:
# === ITC baseline similarity matrix (Q × G) — for ablation, and as score outside top-K ===
Q_t = F.normalize(torch.from_numpy(q_pooled.astype('float32')).to(device), dim=-1)
G_t = F.normalize(torch.from_numpy(g_pooled.astype('float32')).to(device), dim=-1)
S_itc = (Q_t @ G_t.T)  # (Q, G) fp32 on GPU
print('S_itc:', tuple(S_itc.shape))

torch.save({'scores': S_itc.half().cpu(),
            'query_ids': q_ids, 'gallery_ids': g_ids,
            'note': 'ITC-only (cosine of pooled feats) — ablation baseline'},
           UIT_FEAT_DIR / 'scores_uit_itc_only.pt')
print('Saved scores_uit_itc_only.pt (ablation)')

In [ ]:
# === Algorithm 1 (Feature Selection): LHP scores → top-K candidates per query ===
#
# Paper §3.4: LHP scoring matrix acts as guidance to select top-K image candidates.
# Then top-K is passed to the UIT cross-encoder for ITM rerank.
ITM_TOPK = 256
ITM_FUSION_ALPHA = 0.4  # final_topk = α · ITC + (1−α) · ITM

lhp_payload = torch.load(LHP_SCORES, map_location='cpu')
lhp_S = lhp_payload['scores'].float()
lhp_q_ids = np.asarray([str(x) for x in lhp_payload['query_ids']])
lhp_g_ids = np.asarray([str(x) for x in lhp_payload['gallery_ids']])

# Align LHP query/gallery IDs with UIT canonical order (q_ids, g_ids)
lhp_qi = {x: i for i, x in enumerate(lhp_q_ids)}
lhp_gi = {x: i for i, x in enumerate(lhp_g_ids)}
q_perm = np.array([lhp_qi[q] for q in q_ids], dtype=np.int64)
g_perm = np.array([lhp_gi[g] for g in g_ids], dtype=np.int64)
lhp_S_aligned = lhp_S[q_perm][:, g_perm].to(device)
print('LHP aligned:', tuple(lhp_S_aligned.shape))

# Top-K candidate indices per query (in canonical gallery order)
lhp_topvals, lhp_topidx = torch.topk(lhp_S_aligned, ITM_TOPK, dim=1)
print(f'Algorithm 1: chose top-{ITM_TOPK} candidates per query from LHP guidance')

In [ ]:
# === ITM Rerank: UIT Cross-Encoder + itm_head on top-K candidates ===
#
# For each query q:
#   text_embeds = q_tokens[q]
#   for each candidate g in lhp_topidx[q]:
#       cross = get_cross_embeds(image_embeds_g, image_atts_g, text_embeds, text_atts)
#       logits = itm_head(cross[:, 0])           # (2,) — [no_match, match]
#       p_match = softmax(logits)[1]
#
# Batch over gallery axis for each query: shape (K, T_v, D_v) image_embeds + (K, T_v) atts +
# broadcasted (K, T_t, D_t) text_embeds + (K, T_t) text_atts.

ITM_BATCH = 32     # per-query, batch K=256 candidates through the cross-encoder ITM in sub-batches 32
T_v = g_tokens.shape[1]; D_v = g_tokens.shape[2]
T_t = q_tokens.shape[1]; D_t = q_tokens.shape[2]
print(f'image_embeds: T_v={T_v} D_v={D_v} | text_embeds: T_t={T_t} D_t={D_t}')

@torch.inference_mode()
def itm_score_topk(q_idx, cand_idx):
    """Compute P(match) for query q vs top-K gallery candidates.

    q_idx: int
    cand_idx: torch.LongTensor (K,) — gallery indices in canonical order
    Returns: torch.FloatTensor (K,)
    """
    K = cand_idx.shape[0]
    txt_e = q_tokens[q_idx:q_idx+1].to(device, dtype=torch.float32)
    txt_a = q_atts[q_idx:q_idx+1].to(device)
    probs = torch.empty(K, device=device)
    for s in range(0, K, ITM_BATCH):
        e = min(s + ITM_BATCH, K)
        idx_chunk = cand_idx[s:e]
        img_e = g_tokens[idx_chunk].to(device, dtype=torch.float32)
        img_a = g_atts[idx_chunk].to(device)
        txt_e_rep = txt_e.expand(e - s, -1, -1)
        txt_a_rep = txt_a.expand(e - s, -1)
        with torch.autocast(device_type=device.type, dtype=amp_dtype):
            cross = model.get_cross_embeds(img_e, img_a, txt_e_rep, txt_a_rep)
            logits = model.itm_head(cross[:, 0])  # (B, 2)
        probs[s:e] = F.softmax(logits.float(), dim=1)[:, 1]
    return probs

# Run ITM rerank for all queries
S_final = S_itc.clone()  # start from ITC, will overwrite top-K positions
for q in tqdm(range(len(q_ids)), desc='UIT ITM rerank'):
    cand = lhp_topidx[q]                                # (K,) gallery indices
    itc_at_topk = S_itc[q, cand].float()
    itm_at_topk = itm_score_topk(q, cand)
    # Normalize each to [0,1] within top-K, then convex combine
    # Note: ITM is already a probability in [0,1]; ITC is cosine in roughly [-1,1].
    # Convex combine on raw scales — works because we re-rank only within top-K.
    fused = ITM_FUSION_ALPHA * itc_at_topk + (1 - ITM_FUSION_ALPHA) * itm_at_topk
    S_final[q, cand] = fused

torch.save({'scores': S_final.half().cpu(),
            'query_ids': q_ids, 'gallery_ids': g_ids,
            'note': f'ITC outside top-{ITM_TOPK} + (α·ITC + (1-α)·ITM) inside top-K (α={ITM_FUSION_ALPHA})',
            'itm_topk': ITM_TOPK, 'itm_alpha': ITM_FUSION_ALPHA},
           UIT_FEAT_DIR / 'scores_uit.pt')
print('Saved scores_uit.pt:', tuple(S_final.shape))

In [ ]:
# === Val scores (in-distribution + zero-shot scene) ===
# Apply same LHP-guided ITM rerank logic.
for name, df in [('val', val_df), ('val_zs', val_zs_df)]:
    if len(df) == 0:
        continue
    print(f'--- {name} ({len(df)} rows) ---')
    v_pooled, v_tokens, v_atts, v_ids, v_paths, v_ok = encode_images_full(df, 'image_id', keep_tokens=True)
    v_txt_pooled, v_txt_tokens, v_txt_atts, _ = encode_texts_full(
        df['image_id'].astype(str).tolist(), df['caption'].astype(str).tolist(),
        keep_tokens=True,
    )
    save_npz_atomic(UIT_FEAT_DIR / f'{name}.npz', ids=v_ids, paths=v_paths, embeddings=v_pooled, ok=v_ok)
    save_npz_atomic(UIT_FEAT_DIR / f'{name}_text.npz', ids=v_ids, embeddings=v_txt_pooled)

    Qv = F.normalize(torch.from_numpy(v_txt_pooled.astype('float32')).to(device), dim=-1)
    Gv = F.normalize(torch.from_numpy(v_pooled.astype('float32')).to(device), dim=-1)
    S_itc_v = Qv @ Gv.T

    # LHP val scores from 03 (if available)
    lhp_val_path = LHP_FEAT_DIR / f'scores_{name}.pt' if (LHP_FEAT_DIR / f'scores_{name}.pt').exists() else None
    if lhp_val_path is None:
        # Fallback: use LHP-PE-G14 embeddings to build val LHP scores on the fly
        lhp_img_npz = LHP_FEAT_DIR / f'{name}.npz'
        lhp_txt_npz = LHP_FEAT_DIR / f'{name}_text.npz'
        if lhp_img_npz.exists() and lhp_txt_npz.exists():
            li = np.load(lhp_img_npz); lt = np.load(lhp_txt_npz)
            Qlhp = F.normalize(torch.from_numpy(lt['embeddings'].astype('float32')).to(device), dim=-1)
            Glhp = F.normalize(torch.from_numpy(li['embeddings'].astype('float32')).to(device), dim=-1)
            S_lhp_v = Qlhp @ Glhp.T
        else:
            print(f'  ⚠️  No LHP val scores for {name} — fallback to ITC only')
            torch.save({'scores': S_itc_v.half().cpu(), 'query_ids': v_ids, 'gallery_ids': v_ids},
                       UIT_FEAT_DIR / f'scores_{name}.pt')
            continue
    else:
        lhp_payload_v = torch.load(lhp_val_path, map_location='cpu')
        S_lhp_v = lhp_payload_v['scores'].float().to(device)

    Kv = min(ITM_TOPK, S_lhp_v.shape[1])
    _, top_idx_v = torch.topk(S_lhp_v, Kv, dim=1)

    @torch.inference_mode()
    def itm_score_val(q_idx, cand_idx, txt_t, txt_a, img_t, img_a):
        K = cand_idx.shape[0]
        probs = torch.empty(K, device=device)
        txt_e = txt_t[q_idx:q_idx+1].to(device, dtype=torch.float32)
        ta = txt_a[q_idx:q_idx+1].to(device)
        for s in range(0, K, ITM_BATCH):
            e = min(s + ITM_BATCH, K)
            idx_chunk = cand_idx[s:e]
            ie = img_t[idx_chunk].to(device, dtype=torch.float32)
            ia = img_a[idx_chunk].to(device)
            te = txt_e.expand(e - s, -1, -1); tax = ta.expand(e - s, -1)
            with torch.autocast(device_type=device.type, dtype=amp_dtype):
                cross = model.get_cross_embeds(ie, ia, te, tax)
                logits = model.itm_head(cross[:, 0])
            probs[s:e] = F.softmax(logits.float(), dim=1)[:, 1]
        return probs

    S_final_v = S_itc_v.clone()
    for qi in tqdm(range(len(v_ids)), desc=f'UIT ITM rerank ({name})'):
        cand = top_idx_v[qi]
        itc_at = S_itc_v[qi, cand].float()
        itm_at = itm_score_val(qi, cand, v_txt_tokens, v_txt_atts, v_tokens, v_atts)
        fused = ITM_FUSION_ALPHA * itc_at + (1 - ITM_FUSION_ALPHA) * itm_at
        S_final_v[qi, cand] = fused

    torch.save({'scores': S_final_v.half().cpu(),
                'query_ids': v_ids, 'gallery_ids': v_ids,
                'itm_topk': Kv, 'itm_alpha': ITM_FUSION_ALPHA},
               UIT_FEAT_DIR / f'scores_{name}.pt')
    print(f'Saved scores_{name}.pt: {tuple(S_final_v.shape)}')
    del v_tokens, v_atts, v_txt_tokens, v_txt_atts; maybe_clear_cuda()

In [ ]:
# === Cleanup + Drive sync ===
# Free token caches (heavy — gallery tokens were ~3.7GB)
del g_tokens, g_atts, q_tokens, q_atts
maybe_clear_cuda()

for f in UIT_FEAT_DIR.rglob('*'):
    if f.is_file():
        sync_chunk_to_drive(f, LOCAL_ROOT, DRIVE_OUTPUT_ROOT, background=True)
wait_for_pending_syncs()
print('UIT inference (LHP-guided + ITM rerank) done.')